# Ensemble Retriever — 하이브리드 검색

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **Sparse(BM25)**와 **Dense(벡터)** 검색 방식의 차이와 각각의 강점 이해하기
- `EnsembleRetriever`로 두 방식을 결합하는 방법 익히기
- `weights` 파라미터로 가중치를 조정해 검색 성능 튜닝하기

## 사전 지식

- BM25: 키워드 빈도를 활용한 전통적 검색 방식
- Dense Retrieval: 임베딩 벡터 간 유사도 기반 검색

---

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> B[BM25<br/>키워드 검색]:::process
    Q --> V[FAISS<br/>의미 검색]:::process
    B -- 가중치 0.5 --> E[EnsembleRetriever<br/>RRF 알고리즘]:::process
    V -- 가중치 0.5 --> E
    E --> R[최적 검색 결과]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## 핵심 아이디어

| 특성 | Sparse / BM25 | Dense / Vector |
|------|:---:|:---:|
| 기반 | 키워드 매칭 | 의미 유사도 |
| 강점 | 정확한 용어 매칭 | 문맥·동의어 이해 |
| 약점 | 동의어 못 찾음 | 키워드 민감도 낮음 |

두 방식을 결합하면 각각의 약점을 보완해요. **RRF(Reciprocal Rank Fusion)** 알고리즘으로 결과를 통합합니다:

```
score = Σ(weight_i / (k + rank_i))
```

> 🎯 **강의 포인트**: EnsembleRetriever는 현업에서 가장 많이 쓰이는 검색 패턴 중 하나입니다. 벡터 검색만으로는 "GPT-4"처럼 정확한 고유명사 검색이 약하고, BM25만으로는 동의어를 처리하지 못해요. 두 방식을 결합하면 서로의 약점을 보완합니다.

> 🔑 **핵심 개념**: RRF(Reciprocal Rank Fusion) 알고리즘은 각 Retriever의 순위를 역수로 변환해 합산합니다. 한 쪽에서만 높은 순위여도 최종 결과에 반영되므로, 어떤 방식도 완전히 무시되지 않아요.

## 환경 설정


In [1]:
from dotenv import load_dotenv

load_dotenv()


True

## 1. 문서 준비 및 개별 Retriever 생성


In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------------------------------------------------
# 하이브리드 검색을 위한 문서 준비
# ---------------------------------------------------

# ============================================================
# TODO: ai-story.txt를 로드하고 500자 청크로 분할하세요
# 힌트: TextLoader → load() → RecursiveCharacterTextSplitter → split_documents()
# 예상 결과: 분할된 청크 수 출력
# ============================================================

# 문서 로드 및 분할
loader = TextLoader("./data/ai-story.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = text_splitter.split_documents(documents)

print(f"✅ 문서 준비 완료: {len(split_docs)}개 청크")

✅ 문서 준비 완료: 19개 청크


In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# 1. BM25 Retriever (Sparse - 키워드 기반)
bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = 3

# 2. FAISS Retriever (Dense - 의미 기반)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ BM25 Retriever 생성 완료 (키워드 기반)")
print("✅ FAISS Retriever 생성 완료 (의미 기반)")


✅ BM25 Retriever 생성 완료 (키워드 기반)
✅ FAISS Retriever 생성 완료 (의미 기반)


In [ ]:
# ---------------------------------------------------
# 개별 Retriever로 동일 쿼리를 검색하여 차이점 확인
# ---------------------------------------------------

# ============================================================
# TODO: bm25_retriever와 faiss_retriever에 동일한 쿼리를 각각 실행하세요
# 힌트: retriever.invoke(query) 후 결과 출력
# 예상 결과: BM25는 키워드 매칭, FAISS는 의미 유사도 기반 문서 반환
# ============================================================

# 개별 retriever 성능 비교
query = "딥러닝 모델에서 가중치 학습은 어떻게 이루어지나요?"

bm25_docs = bm25_retriever.invoke(query)
faiss_docs = faiss_retriever.invoke(query)

print(f"📝 쿼리: {query}\n")
print("="*80)
print("[BM25 결과 - 키워드 기반]")
print("="*80)
for i, doc in enumerate(bm25_docs, 1):
    print(f"[{i}] {doc.page_content[:100]}...")
    print()

print("="*80)
print("[FAISS 결과 - 의미 기반]")
print("="*80)
for i, doc in enumerate(faiss_docs, 1):
    print(f"[{i}] {doc.page_content[:100]}...")
    print()

print("\n💡 관찰:")
print("  - BM25: '학습', '가중치' 등 키워드가 명확한 문서 우선")
print("  - FAISS: 전체 문맥과 의미가 유사한 문서 우선")

📝 쿼리: 딥러닝 모델에서 가중치 학습은 어떻게 이루어지나요?

[BM25 결과 - 키워드 기반]
[1] 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Atte...

[2] Word2Vec의 벡터 표현은 다양한 NLP 작업에 활용될 수 있습니다. 예를 들어, 단어의 유사도 측정, 문장이나 문서의 벡터 표현 생성, 기계 번역, 감정 분석 등이 있습니다....

[3] Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니다. 이에는 텍스트 분류, 정보 추출, 질문 응답, 요약 생성, 텍스트 생성 등이...

[FAISS 결과 - 의미 기반]
[1] 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Atte...

[2] NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 복잡하고, 문맥에 따라 그 의미가 달라질 수 있으며, 같은 단어나 구문이 다양한 의미로 사...

[3] 핵심 특징 중 하나는 다양한 기계 학습 모델을 일관된 인터페이스로 제공한다는 점입니다. 이는 사용자가 알고리즘 간에 쉽게 전환할 수 있게 하여, 최적의 모델을 찾는 과정을 단순화합...


💡 관찰:
  - BM25: '학습', '가중치' 등 키워드가 명확한 문서 우선
  - FAISS: 전체 문맥과 의미가 유사한 문서 우선


## 2. EnsembleRetriever 생성 및 사용

두 retriever를 결합하여 각각의 강점을 활용합니다.


In [5]:
from langchain.retrievers import EnsembleRetriever

# EnsembleRetriever 생성 (가중치 5:5)
# 🎯 강의 포인트: weights 합은 반드시 1.0이어야 함 (0.5+0.5=1.0)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5]
)

# Ensemble 검색
ensemble_docs = ensemble_retriever.invoke(query)

print(f"📝 쿼리: {query}\n")
print("="*80)
print("[Ensemble 결과 - 하이브리드 검색]")
print("="*80)
for i, doc in enumerate(ensemble_docs, 1):
    print(f"[{i}] {doc.page_content[:100]}...")
    print()

print("\n💡 장점:")
print("  - BM25와 FAISS의 결과를 모두 반영")
print("  - RRF 알고리즘으로 최적 순위 결정")
print("  - 더 다양하고 관련성 높은 결과")

📝 쿼리: 딥러닝 모델에서 가중치 학습은 어떻게 이루어지나요?

[Ensemble 결과 - 하이브리드 검색]
[1] 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Atte...

[2] Word2Vec의 벡터 표현은 다양한 NLP 작업에 활용될 수 있습니다. 예를 들어, 단어의 유사도 측정, 문장이나 문서의 벡터 표현 생성, 기계 번역, 감정 분석 등이 있습니다....

[3] NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 복잡하고, 문맥에 따라 그 의미가 달라질 수 있으며, 같은 단어나 구문이 다양한 의미로 사...

[4] Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니다. 이에는 텍스트 분류, 정보 추출, 질문 응답, 요약 생성, 텍스트 생성 등이...

[5] 핵심 특징 중 하나는 다양한 기계 학습 모델을 일관된 인터페이스로 제공한다는 점입니다. 이는 사용자가 알고리즘 간에 쉽게 전환할 수 있게 하여, 최적의 모델을 찾는 과정을 단순화합...


💡 장점:
  - BM25와 FAISS의 결과를 모두 반영
  - RRF 알고리즘으로 최적 순위 결정
  - 더 다양하고 관련성 높은 결과


In [6]:
# ---------------------------------------------------
# EnsembleRetriever의 가중치를 변경하여 결과 차이 실험
# ---------------------------------------------------

# ============================================================
# TODO: BM25 우선(7:3)과 FAISS 우선(3:7) 두 가지 가중치로 같은 쿼리를 검색하세요
# 힌트: EnsembleRetriever(retrievers=[...], weights=[w1, w2])
# 예상 결과: 가중치에 따라 상위 문서 순서가 달라짐
# ============================================================

# 가중치 실험
test_query = "Transformer 아키텍처의 핵심 요소는?"

# 케이스 1: BM25 우선 (7:3)
# 전문 용어가 많거나 정확한 키워드 매칭이 중요할 때
ensemble_bm25_heavy = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.7, 0.3]
)

# 케이스 2: FAISS 우선 (3:7)
# 자연어 질문이 많거나 동의어를 고려해야 할 때
ensemble_faiss_heavy = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.3, 0.7]
)

docs_bm25_heavy = ensemble_bm25_heavy.invoke(test_query)
docs_faiss_heavy = ensemble_faiss_heavy.invoke(test_query)

print(f"📝 쿼리: {test_query}\n")
print("="*80)
print("[BM25 우선 (7:3) - 키워드 중시]")
print("="*80)
for i, doc in enumerate(docs_bm25_heavy[:2], 1):
    print(f"[{i}] {doc.page_content[:80]}...")
print()

print("="*80)
print("[FAISS 우선 (3:7) - 의미 중시]")
print("="*80)
for i, doc in enumerate(docs_faiss_heavy[:2], 1):
    print(f"[{i}] {doc.page_content[:80]}...")
print()

print("\n💡 가중치 선택 가이드:")
print("  - 전문 용어 많음 → BM25 가중치 높임 (0.6-0.7)")
print("  - 자연어 질문 → FAISS 가중치 높임 (0.6-0.7)")
print("  - 일반적 사용 → 균등 (0.5-0.5)")

📝 쿼리: Transformer 아키텍처의 핵심 요소는?

[BM25 우선 (7:3) - 키워드 중시]
[1] Scikit Learn

Scikit-learn은 Python 언어를 위한 또 다른 핵심 라이브러리로, 기계 학습의 다양한 알고리즘을 구현하기 ...
[2] 핵심 특징 중 하나는 다양한 기계 학습 모델을 일관된 인터페이스로 제공한다는 점입니다. 이는 사용자가 알고리즘 간에 쉽게 전환할 수 있게 하여,...

[FAISS 우선 (3:7) - 의미 중시]
[1] Attention is all you need

"Attention Is All You Need"는 2017년에 발표된 논문으로, 변환기(Tra...
[2] "Attention Is All You Need" 논문은 변환기 아키텍처를 기반으로 한 '멀티헤드 어텐션(Multi-Head Attention)...


💡 가중치 선택 가이드:
  - 전문 용어 많음 → BM25 가중치 높임 (0.6-0.7)
  - 자연어 질문 → FAISS 가중치 높임 (0.6-0.7)
  - 일반적 사용 → 균등 (0.5-0.5)


In [7]:
# ---------------------------------------------------
# 가중치 변화에 따른 결과 비교 + 문서 출처 추적
# ---------------------------------------------------

# 동일 쿼리에 대해 가중치 3가지를 체계적으로 비교
weight_configs = [
    ([0.7, 0.3], "BM25 우선 (7:3)"),
    ([0.5, 0.5], "균등 (5:5)"),
    ([0.3, 0.7], "FAISS 우선 (3:7)"),
]

compare_query = "Transformer 아키텍처의 핵심 요소는?"

# 각 가중치별 결과 수집
bm25_results = set(d.page_content[:60] for d in bm25_retriever.invoke(compare_query))
faiss_results = set(d.page_content[:60] for d in faiss_retriever.invoke(compare_query))

print(f"📝 쿼리: {compare_query}\n")
print(f"{'='*80}")
print(f"{'순위':<5} {'BM25 우선 (7:3)':<25} {'균등 (5:5)':<25} {'FAISS 우선 (3:7)':<25}")
print(f"{'='*80}")

# 각 가중치별 결과 저장
all_results = {}
for weights, label in weight_configs:
    retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, faiss_retriever],
        weights=weights
    )
    all_results[label] = retriever.invoke(compare_query)

# 순위 비교 테이블 (상위 5개)
max_docs = max(len(v) for v in all_results.values())
for rank in range(min(max_docs, 5)):
    row = []
    for _, label in weight_configs:
        docs = all_results[label]
        if rank < len(docs):
            snippet = docs[rank].page_content[:20] + "..."
        else:
            snippet = "-"
        row.append(snippet)
    print(f"[{rank+1}]  {row[0]:<25} {row[1]:<25} {row[2]:<25}")

# 문서 출처 분석
print(f"\n{'='*80}")
print("📊 문서 출처 분석 (균등 5:5 기준)")
print(f"{'='*80}")
balanced_docs = all_results["균등 (5:5)"]
for i, doc in enumerate(balanced_docs[:5], 1):
    key = doc.page_content[:60]
    from_bm25 = "✅" if key in bm25_results else "❌"
    from_faiss = "✅" if key in faiss_results else "❌"
    print(f"[{i}] BM25:{from_bm25} FAISS:{from_faiss} | {doc.page_content[:50]}...")

print("\n💡 관찰 포인트:")
print("  - 가중치에 따라 상위 문서 순서가 달라짐")
print("  - 양쪽 모두에서 검색된 문서(✅✅)가 상위에 위치하는 경향")
print("  - 한쪽에서만 검색된 문서는 해당 방식의 가중치가 높을 때 상위로 올라감")

📝 쿼리: Transformer 아키텍처의 핵심 요소는?

순위    BM25 우선 (7:3)             균등 (5:5)                  FAISS 우선 (3:7)           
[1]  Scikit Learn

Scikit...   Scikit Learn

Scikit...   Attention is all you...  
[2]  핵심 특징 중 하나는 다양한 기계 학...   Attention is all you...   "Attention Is All Yo...  
[3]  Word2Vec

Word2Vec은 ...   핵심 특징 중 하나는 다양한 기계 학...   어텐션 메커니즘은 입력 데이터의 서로...  
[4]  Attention is all you...   "Attention Is All Yo...   Scikit Learn

Scikit...  
[5]  "Attention Is All Yo...   Word2Vec

Word2Vec은 ...   핵심 특징 중 하나는 다양한 기계 학...  

📊 문서 출처 분석 (균등 5:5 기준)
[1] BM25:✅ FAISS:❌ | Scikit Learn

Scikit-learn은 Python 언어를 위한 또 다른 핵심 ...
[2] BM25:❌ FAISS:✅ | Attention is all you need

"Attention Is All You N...
[3] BM25:✅ FAISS:❌ | 핵심 특징 중 하나는 다양한 기계 학습 모델을 일관된 인터페이스로 제공한다는 점입니다. 이...
[4] BM25:❌ FAISS:✅ | "Attention Is All You Need" 논문은 변환기 아키텍처를 기반으로 한 '...
[5] BM25:✅ FAISS:❌ | Word2Vec

Word2Vec은 자연어 처리(NLP) 분야에서 널리 사용되는 획기적인 ...

💡 관찰 포인트:
  - 가중치에 따라 상위 문서 순서가 달라짐
  - 양쪽 모두에서 검색된 문서(✅✅)가 상위에 위치하는

## 마무리 정리

이 노트북에서 배운 내용을 정리해요:

| 가중치 설정 | 상황 | 예시 |
|-----------|------|------|
| `[0.7, 0.3]` | 전문 용어·고유명사 중심 | 의학, 법률 문서 |
| `[0.5, 0.5]` | 일반적 사용 (기본 권장) | 뉴스, 블로그 |
| `[0.3, 0.7]` | 자연어 질문·동의어 많음 | 고객 상담, Q&A |

> 💡 **실무 팁**: 가중치 최적값은 데이터와 쿼리 유형에 따라 달라집니다. 검색 품질을 평가할 수 있는 소규모 테스트 셋을 만들어 두고, 여러 가중치 조합을 비교하는 것이 좋아요.

> ⚠️ **자주 하는 실수**: BM25Retriever의 `k`와 FAISS Retriever의 `k`를 동일하게 맞추지 않으면 가중치의 실제 효과가 불균형해집니다. 두 Retriever의 k는 같게 설정하세요.

### 다음 노트북 예고

**04-LongContextReorder**: LLM이 긴 컨텍스트의 중간 부분을 잘 활용하지 못하는 "Lost-in-the-Middle" 문제와 해결법을 배워요.